# Build standard DPO pairs from strict-extractive candidate banks

Construct ordinary pairwise DPO data from the new 0.5B/3B strict-extractive banks. The selected bank can be `05b`, `3b`, or `both`. Only the train split is used for training pairs; dev and test remain evaluation-only.

In [ ]:
from pathlib import Path
import json, shutil, subprocess
PROJECT_ROOT=Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
SPLIT='train'
BANK_SELECTION='05b'  # '05b', '3b', or 'both'
SEED=3407
WRONGS_PER_QUESTION=1000000  # Effectively unlimited: retain every valid candidate.
REQUIRE_WITHIN_QUESTION_WRONGS=False  # Keep questions with at least one valid negative.
RUN_ROOT=PROJECT_ROOT/'Artifacts/cse_dpo/candidate_banks/strict_extractive_qwen25_05b_3b_all_splits_s20_v2_seed3407'
BANK_ROOTS={'05b':RUN_ROOT/'qwen25_05b'/SPLIT/'adapter-best-evidence-mrr/candidate_bank.jsonl','3b':RUN_ROOT/'qwen25_3b'/SPLIT/'adapter-best-evidence-mrr/candidate_bank.jsonl'}
QUESTION_INPUT=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/single_answer_full_resources/train_prepared.json'
OUTPUT_ROOT=PROJECT_ROOT/'Artifacts/cse_dpo/standard_dpo_pairs_from_strict_extractive_banks'/f'{SPLIT}_{BANK_SELECTION}_all_valid_negatives_seed{SEED}'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
assert BANK_SELECTION in {'05b','3b','both'}
assert QUESTION_INPUT.exists()
for key in (('05b','3b') if BANK_SELECTION=='both' else (BANK_SELECTION,)): assert BANK_ROOTS[key].exists(), BANK_ROOTS[key]
print('Selection:',BANK_SELECTION,'| question input:',QUESTION_INPUT,'| output:',OUTPUT_ROOT)

## Merge the selected model banks

The constructor accepts one candidate-bank JSONL. When `both` is selected, this cell merges both model banks while preserving model provenance and namespacing response IDs.

In [ ]:
selected=('05b','3b') if BANK_SELECTION=='both' else (BANK_SELECTION,)
MERGED_BANK=OUTPUT_ROOT/'candidate_bank.jsonl'
count=0
with MERGED_BANK.open('w',encoding='utf-8') as out:
    for key in selected:
        for line in BANK_ROOTS[key].open(encoding='utf-8'):
            if not line.strip(): continue
            row=json.loads(line); source=row['response_id']
            row['source_response_id']=source; row['response_id']=f'{key}::{source}'; row['generator_model_size']=key
            out.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+'\n'); count+=1
print('Merged candidates:',count,'at',MERGED_BANK)

## Construct chosen/rejected pairs

The builder treats the prepared supported alias as `chosen` and samples distinct non-gold entity-like generations as `rejected`. With `REQUIRE_WITHIN_QUESTION_WRONGS=False`, every valid within-question negative is retained and questions with no valid negative are skipped; no global backfill is used.

In [ ]:
pairs=OUTPUT_ROOT/'pairs.jsonl'; classes=OUTPUT_ROOT/'question_classes.jsonl'; summary=OUTPUT_ROOT/'summary.json'
cmd=['python','-m','cse_dpo.construct_factoid_label_wrong_entity_bank','--question-input',str(QUESTION_INPUT),'--candidate-bank-jsonl',str(MERGED_BANK),'--output-jsonl',str(classes),'--pair-output-jsonl',str(pairs),'--summary-json',str(summary),'--wrongs-per-question',str(WRONGS_PER_QUESTION),'--max-entity-chars','160','--max-entity-words','14','--seed',str(SEED)]
cmd += ['--require-within-question-wrongs'] if REQUIRE_WITHIN_QUESTION_WRONGS else ['--allow-fewer-within-question-wrongs']
print(' '.join(cmd)); subprocess.run(cmd,cwd=PROJECT_ROOT,check=True)
print(json.dumps(json.loads(summary.read_text()),indent=2))

In [ ]:
# Validate and preview the resulting standard-DPO data.
pair_rows=[json.loads(x) for x in pairs.open(encoding='utf-8') if x.strip()]
class_rows=[json.loads(x) for x in classes.open(encoding='utf-8') if x.strip()]
assert pair_rows and class_rows
assert all(r.get('chosen') and r.get('rejected') for r in pair_rows)
assert all(r.get('split',SPLIT)==SPLIT for r in pair_rows)
print('Questions:',len(class_rows),'| pair rows:',len(pair_rows))
for r in pair_rows[:5]: print(r['question_id'], 'chosen=',r['chosen'], 'rejected=',r['rejected'], 'source=',r.get('rejected_source'))
manifest={'split':SPLIT,'bank_selection':BANK_SELECTION,'seed':SEED,'wrongs_per_question':WRONGS_PER_QUESTION,'question_input':str(QUESTION_INPUT),'candidate_bank':str(MERGED_BANK),'classes':str(classes),'pairs':str(pairs),'pair_count':len(pair_rows),'question_count':len(class_rows)}
(OUTPUT_ROOT/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('Manifest:',OUTPUT_ROOT/'manifest.json')